# N172 · Medium综合题与机制组合

**目标：** 在两个已知机制之间定义清楚接口。

**先修：** N061, N094, N097, N035。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 排序+堆；前缀+哈希；单调结构+贪心；分离状态职责。

## 从问题到算法

三个问题共享“局部选择＋结构约束”，但不应共用一个未经证明的贪心模板。

任务冷却的下界一是总任务数，二是把最高频任务的前 f−1 次隔开形成长度 n+1 的框架，再放并列最高频的末次任务，得到 `(f−1)(n+1)+c`。两者最大值可由排列填空达到；演示另构造真实冷却时间线核验。

字符重排禁止相邻重复，暂时扣住刚输出字符，再从剩余最大频字符中选择。视频覆盖则在当前可接上的所有片段中选择结束最远的，更新覆盖前沿；片段的输入顺序无关。

## 最小实现

**本章接口：** `least_interval(tasks, n)`、`reorganize_string(s)`、`video_stitching(clips, time)`

In [1]:
from collections import Counter, deque
import heapq

def least_interval(tasks,n):
    if n<0:raise ValueError('Cooldown must be nonnegative')
    counts=Counter(tasks)
    if not counts:return 0
    maximum=max(counts.values())
    tied=sum(value==maximum for value in counts.values())
    return max(len(tasks),(maximum-1)*(n+1)+tied)

def cooldown_schedule(tasks,n):
    """Small-case trace with None for idle slots; labels must be comparable."""
    if n<0:raise ValueError('Cooldown must be nonnegative')
    ready=[(-count,label) for label,count in Counter(tasks).items()]
    heapq.heapify(ready)
    cooling=deque();trace=[];time=0
    while ready or cooling:
        while cooling and cooling[0][0]<=time:
            _,count,label=cooling.popleft();heapq.heappush(ready,(count,label))
        if ready:
            count,label=heapq.heappop(ready);trace.append(label)
            if count+1<0:cooling.append((time+n+1,count+1,label))
        else:trace.append(None)
        time+=1
    return trace

def reorganize_string(s):
    counts=Counter(s)
    if max(counts.values(),default=0)>(len(s)+1)//2:return ''
    heap=[(-count,char) for char,count in counts.items()];heapq.heapify(heap)
    held=(0,'');out=[]
    while heap:
        count,char=heapq.heappop(heap);out.append(char)
        if held[0]<0:heapq.heappush(heap,held)
        held=(count+1,char)
    return ''.join(out) if held[0]==0 else ''

def video_stitching(clips,time):
    if time<0:raise ValueError('Nonnegative target time required')
    clips=sorted(clips);i=answer=0;covered=0
    while covered<time:
        farthest=covered
        while i<len(clips) and clips[i][0]<=covered:
            farthest=max(farthest,clips[i][1]);i+=1
        if farthest==covered:return -1
        covered=farthest;answer+=1
    return answer

## 正确性、前提与复杂度

冷却框架给出必要下界：最后一轮之前，每个最高频任务之间都要留足间隔；其他任务可填入空位，若超出空位则无须空闲。重排中最大频率不超过其余字符间隙数是可行必要条件，优先消耗最多者避免把稀缺间隙留到最后。区间贪心用交换论证：用结束更远、同样可接上的片段替换任一首段，不增加段数且不缩短后续可达范围。

**代价：** 冷却最短长度计算 O(m)，不同标签空间 O(u)；演示时间线可能包含大量空闲，仅用于小输入。重排 O(m log u)，覆盖排序 O(k log k)、扫描 O(k)。

## 小实例与可视化

先手算，再运行。下表由当前内核中的代码和显式小实例生成；不是预制答案图片。

In [2]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

trace=cooldown_schedule('AAABBB',2)
show_table(['time','task'],list(enumerate(trace)))
show_table(['problem','input','result'],[
 ['reorganization','aaabbc',reorganize_string('aaabbc')],
 ['coverage',[[0,2],[1,5],[4,8]],video_stitching([[0,2],[1,5],[4,8]],8)]])

time,task
0,A
1,B
2,None
3,A
4,B
5,None
6,A
7,B


problem,input,result
reorganization,aaabbc,ababac
coverage,"[[0, 2], [1, 5], [4, 8]]",3


## 自动测试

以下断言检查实现契约。测试通过不等同于一般性证明，也不表示已经提交 LeetCode 官方评测。

In [3]:
from itertools import product,combinations
from random import Random
assert least_interval('AAABBB',2)==8
assert reorganize_string('aaab')==''
assert video_stitching([[0,1],[2,3]],3)==-1
assert video_stitching([],0)==0
def brute_cooldown(counts,n):
    start=(tuple(counts),(0,)*len(counts));queue=deque([(start,0)]);seen={start}
    while queue:
        (remaining,wait),time=queue.popleft()
        if not any(remaining):return time
        choices=[i for i in range(len(counts)) if remaining[i] and wait[i]==0]+[None]
        for choice in choices:
            new_counts=list(remaining);new_wait=[max(0,x-1) for x in wait]
            if choice is not None:new_counts[choice]-=1;new_wait[choice]=n
            state=(tuple(new_counts),tuple(new_wait))
            if state not in seen:seen.add(state);queue.append((state,time+1))
for counts in product(range(3),repeat=3):
    tasks=''.join(chr(65+i)*count for i,count in enumerate(counts))
    for n in range(3):
        expected=brute_cooldown(counts,n)
        assert least_interval(tasks,n)==expected
        trace=cooldown_schedule(tasks,n)
        assert len(trace)==expected and Counter(x for x in trace if x is not None)==Counter(tasks)
        last={}
        for time,label in enumerate(trace):
            if label is not None:
                assert label not in last or time-last[label]>n
                last[label]=time
    out=reorganize_string(tasks)
    possible=max(counts,default=0)<=(len(tasks)+1)//2
    if possible:assert Counter(out)==Counter(tasks) and all(a!=b for a,b in zip(out,out[1:]))
    else:assert out==''
def brute_cover(clips,target):
    for count in range(len(clips)+1):
        for subset in combinations(clips,count):
            end=0
            for left,right in sorted(subset):
                if left>end:break
                end=max(end,right)
            if end>=target:return count
    return -1
rng=Random(172)
for _ in range(250):
    clips=[]
    for i in range(rng.randrange(8)):
        left=rng.randrange(7);clips.append((left,left+rng.randrange(1,5)))
    target=rng.randrange(10)
    assert video_stitching(clips,target)==brute_cover(clips,target)
print('N172: 所有本章断言通过')

N172: 所有本章断言通过


## 练习：先独立完成

**练习 1：** 同题实现计数公式与事件模拟。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

**练习 2：** 解释正确组合而非无依据堆叠模板。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

未完成练习不阻断教学区 Run All。验收时解释为什么正确，不只展示运行截图。

## 代表题与迁移

- 621. Task Scheduler（canonical）
- 767. Reorganize String（canonical）
- 1024. Video Stitching（canonical）

这些题用于知识映射；本章实现遵循上文明确的教学契约。除原规格注明已核对身份的条目外，本轮未重新联网核验全部题面、收费权限或平台语言支持。不要把同概念教学实例当作所有官方题的完整答案。

## 自测与下一步

不看代码，复述状态、不变量和一个失效条件；改变一个输入约束，判断实现是否仍成立。确认能解释边界测试后，按 `specs/curriculum.json` 的 `learning_order` 进入下一章。